# 01 - Análise exploratória
Objetivo: entender o alvo `IN_ALFABETIZADO`, identificar *leakage* e orientar a modelagem.

_Gerado por Claude._

In [ ]:
import sys; sys.path.insert(0, '..')  # permite importar src/ a partir de notebooks/
import pandas as pd, json
from IPython.display import Image, display
from src.config import GOLD, REPORTS, IMAGES

In [ ]:
df = pd.read_parquet(GOLD / 'alunos_gold.parquet')
df.shape, df.groupby('NU_ANO_AVALIACAO').IN_ALFABETIZADO.agg(['size','mean'])

## Hipóteses
1. O desempenho passado da escola/município prevê o do aluno atual.
2. Infraestrutura e financiamento (FUNDEB) têm efeito secundário.
3. Existe forte desigualdade regional.
4. `VL_PROFICIENCIA_LP` define o alvo -> **leakage** (AUC=1.0, corte em 743).

In [ ]:
print(open(REPORTS / 'eda_summary.json').read())

In [ ]:
for f in ['eda_taxa_por_uf','eda_correlacao_alvo','eda_heatmap','eda_hist_escola_lag','eda_hist_mun_lag']:
    display(Image(IMAGES / f'{f}.png'))

In [ ]:
pd.read_csv(REPORTS / 'faltantes_por_ano.csv', index_col=0).sort_values('2024', ascending=False).head(15)

## Decisões de modelagem derivadas da EDA
- Excluir proficiência, pesos, respostas, IDs e agregados do mesmo ano.
- Usar *lags* (ano anterior) e validação temporal (treino 2024, teste 2025).
- Distribuição por níveis municipal só existe a partir de 2024 -> descartada (drift).
- Faltantes -> mediana + indicador de ausência dentro da pipeline.
- `ID_ESCOLA` é anonimizado -> Censo agregado por município x rede.